# Rebuilding drawdown from adjoint sensitivities and superposition

Drawdown at a point is the sum of the drawdowns caused by each pumping well
acting on its own. That is the principle of **superposition**, and it is what
lets an analytical solution such as the Theis equation add up the effect of a
well field. This notebook does the same thing with a MODFLOW 6 model, but takes
each well's response from an **adjoint sensitivity** rather than from an
analytical formula.

By the end of this notebook you will be able to:

- compute the sensitivity of head at an observation well to each pumping well's
  rate with [mf6adj](https://github.com/INTERA-Inc/mf6adj),
- check that superposition against the **Theis** analytical solution on an
  aquifer that meets Theis's assumptions,
- rebuild the drawdown history at that observation well by superposing those
  sensitivities,
- separate the reconstructed drawdown into the share contributed by each well,
  and
- map the reconstructed drawdown over the whole model and compare it with the
  simulated drawdown.

## The idea

The adjoint gives the derivative of a chosen model output with respect to every
parameter, and a well's pumping rate is one of those parameters. For an
observation well, the derivative of its head with respect to a pumping rate,

$$\frac{\partial h}{\partial Q_i},$$

is the **unit response**: the head change that well causes per unit of pumping.
Multiply it by the rate the well actually pumps and add up the wells, and you
have the drawdown:

$$s(t) = -\sum_i \sum_{\tau} \frac{\partial h(t)}{\partial Q_i(\tau)}\, Q_i(\tau).$$

The inner sum over $\tau$ runs over the stress periods, because a rate applied
in an earlier period still affects the head later. Groundwater flow is linear
when the aquifer is confined and the boundaries are fixed, so this sum should
reproduce the simulated drawdown. Where it does not, the difference measures how
far the model departs from linearity.

Import the packages this notebook uses, and locate the MODFLOW 6 executable and
shared library.

In [ ]:
import pathlib as pl

import flopy
import matplotlib.pyplot as plt
import mf6_adj_helpers as adjh
import mf6adj
import numpy as np
from mf6_notebook_helpers import find_mf6_libraries

lib_name, mf6_exe = find_mf6_libraries()

## First, verify against a solution we already know

Before taking the superposition to a real model, check it on a problem with an
exact answer. The **Theis** solution gives the drawdown around a well in a
confined aquifer, and it is itself built on superposition: several wells are
added together in space, and a well that starts later is added in time.

So there are three ways to get the drawdown at a point in such an aquifer, and
all three should agree:

1. run MODFLOW 6 with the wells on and difference it against a run with them off,
2. superpose the adjoint sensitivities, as this notebook does, or
3. evaluate the Theis equation.

Build an aquifer that meets Theis's assumptions — one confined layer,
homogeneous, uniform thickness, fully penetrating wells, and a domain wide
enough that its edges are never felt. The parameters are in
`mf6_adj_helpers.py`; the domain is 30 km across, twice the radius of influence
at the end of the simulation, and the perimeter is held at the starting head,
which is Theis's condition that drawdown vanishes far away.


In [ ]:
print(f"transmissivity T = {adjh.THEIS_T:,.0f} m2/d")
print(f"storativity    S = {adjh.THEIS_S:.4f}")
radius = np.sqrt(
    2.25 * adjh.THEIS_T * adjh.THEIS_NPER * adjh.THEIS_PERLEN / adjh.THEIS_S
)
print(
    f"radius of influence at {adjh.THEIS_NPER * adjh.THEIS_PERLEN:.0f} days: {radius:,.0f} m"
)
print(f"half-width of the domain:      {adjh.THEIS_HALF:,.0f} m")

for name, (x, y, q, start) in adjh.THEIS_WELLS.items():
    print(
        f"  well {name}: {q:8,.0f} m3/d from period {start + 1:2d}"
        f"   at ({x:7,.0f}, {y:7,.0f}) m"
    )

Run the model once with the wells pumping. Three wells start in different
stress periods, so the superposition has to get both the spatial and the
temporal part right.


In [ ]:
theis_ws = pl.Path("models/adj-theis")
theis_sim = adjh.theis_simulation(theis_ws, mf6_exe)
theis_sim.write_simulation(silent=True)
success, buff = theis_sim.run_simulation(silent=True)
assert success, "MODFLOW 6 did not terminate normally"

theis_head = adjh.theis_period_heads(theis_ws)

chd, wel, share = adjh.theis_boundary_share(theis_ws)
print(f"pumped from the wells:      {wel:9,.0f} m3/d")
print(f"drawn from the perimeter:   {chd:9,.0f} m3/d  ({100 * share:.2f}%)")

**What to look for.** By the last period the perimeter is supplying a few per
cent of the pumped water, so the cone has reached it. That water comes from
beyond the observation wells, though, which all sit within 4 km of the pumping:
widening the domain until the perimeter gives up nothing at all changes the
drawdown at those wells by less than a tenth of a millimetre. The drawdown *at*
the perimeter is zero whatever the domain size, because that is the condition
imposed there, so it is the flow across it that is worth looking at.


### One backward solve per well

Use **reciprocity** again: put the performance measure at each *well* rather
than at each observation point, and one backward solve returns that well's
response everywhere in the model. Three wells over ten stress periods is thirty
measures, and from them the drawdown can be rebuilt at any point — the four
observation wells below, or the whole grid.


In [ ]:
theis_measures = {}
for name, (x, y, _, _) in adjh.THEIS_WELLS.items():
    for kper in range(adjh.THEIS_NPER):
        cellid = adjh.theis_cell(x, y)
        theis_measures[f"{name}{kper:02d}"] = [
            (kper, adjh.THEIS_NSTP - 1, 0, *cellid, "head")
        ]

theis_file = adjh.write_adj_file(theis_ws, "theis.adj", theis_measures)
adj = mf6adj.Mf6Adj(
    theis_file.name,
    str(lib_name),
    logging_level="WARNING",
    working_directory=str(theis_ws),
)
adj.solve_forward_model()
adj.solve_adjoint()
adj.finalize()
print(f"solved {len(theis_measures)} performance measures")

Superpose exactly as before: multiply each well's response by the rate it
pumped in each period and add up the wells and the periods.


In [ ]:
theis_rates = adjh.theis_rates()
superposed = {name: np.zeros(adjh.THEIS_NPER) for name in adjh.THEIS_OBS}

for well in adjh.THEIS_WELLS:
    for kper_pm in range(adjh.THEIS_NPER):
        kernels = adjh.period_sensitivity(theis_ws, f"{well}{kper_pm:02d}", "wel6_q")
        for kper, sens in kernels.items():
            for obs, (x, y) in adjh.THEIS_OBS.items():
                cellid = (0, *adjh.theis_cell(x, y))
                superposed[obs][kper_pm] += -sens[cellid] * theis_rates[well][kper]

### Compare all three

Read the simulated drawdown at each observation well, evaluate the Theis
equation there, and put the three side by side.


In [ ]:
days = np.arange(1, adjh.THEIS_NPER + 1) * adjh.THEIS_PERLEN

fig, axes = plt.subplots(2, 2, figsize=(11, 7), sharex=True, constrained_layout=True)
for ax, (obs, (x, y)) in zip(axes.flat, adjh.THEIS_OBS.items(), strict=False):
    row, col = adjh.theis_cell(x, y)
    simulated_obs = -theis_head[:, row, col]
    analytical = adjh.theis_analytical(x, y)

    ax.plot(days, simulated_obs, "o-", color="k", label="MODFLOW 6")
    ax.plot(
        days, superposed[obs], "s--", color="tab:red", ms=5, label="superposed adjoint"
    )
    ax.plot(days, analytical, "-", color="tab:blue", lw=1.2, label="Theis")
    ax.invert_yaxis()
    ax.set_title(f"{obs} — {np.hypot(x, y):,.0f} m from well A", fontsize=10)
    ax.set_ylabel("drawdown (m)")
axes[1, 0].set_xlabel("time (days)")
axes[1, 1].set_xlabel("time (days)")
axes[0, 0].legend(fontsize=8)

worst_adjoint = worst_theis = 0.0
for obs, (x, y) in adjh.THEIS_OBS.items():
    row, col = adjh.theis_cell(x, y)
    simulated_obs = -theis_head[:, row, col]
    worst_adjoint = max(worst_adjoint, np.abs(superposed[obs] - simulated_obs).max())
    worst_theis = max(
        worst_theis, np.abs(adjh.theis_analytical(x, y) - simulated_obs).max()
    )
print(f"largest difference, superposed vs MODFLOW 6: {worst_adjoint:.2e} m")
print(f"largest difference, Theis vs MODFLOW 6:      {worst_theis:.4f} m")

**What to look for.** The three curves lie on top of one another at all four
observation wells, and the steps where wells B and C start are reproduced by
each of them.

The two differences are not the same kind of thing. The superposed adjoint
matches the model to about 1e-11 m — machine precision — because it *is* the
model's own response, rebuilt from the sensitivities rather than recomputed.
Nothing is approximated in that step, so nothing is lost.

Theis differs by up to about two centimetres, and that is the model's
discretization rather than anything wrong with either. MODFLOW 6 reports a head
averaged over a 500 m cell and advances in finite time steps, while Theis is a
point value in continuous time. The time step is what dominates here: raising
`THEIS_NSTP` from 5 to 20 halves the difference, while refining the grid or
widening the domain barely moves it. The difference is largest in the first
period, when the cone of depression is younger than the time step can resolve,
and it shrinks as the cone grows.

That is the check worth having. The adjoint superposition is exact against the
model, and the model converges on the analytical solution as it is refined.


## Now a real model

The idealised aquifer above satisfies every assumption Theis makes. A real one
does not: the synthetic valley has a river, recharge, evapotranspiration,
layers of different conductivity, and wells that only partly penetrate it. The
Theis equation has nothing to say there — but the adjoint superposition still
does, because it takes each well's response from the model itself.


## Prepare the models

Use the **base** synthetic-valley model at an annual sampling frequency. It
represents the river with the River (**RIV**) package and recharge and
evapotranspiration with RCH and EVT, and — importantly here — all of its wells
are ordinary WEL cells, which is the package whose rate the adjoint
differentiates.

Run the model twice. The first run is the model as it ships, with all wells
pumping. The second turns every well off, which gives the head the valley would
have had with no pumping at all. The difference between them is the **simulated
drawdown** that the superposition has to reproduce.

In [ ]:
ws = adjh.prepare_model("adj-drawdown", variant="base")
ws_off = adjh.prepare_model("adj-drawdown-nopump", variant="base", pumping=False)
for w in (ws, ws_off):
    adjh.run_model(w, mf6_exe)

sim = flopy.mf6.MFSimulation.load(sim_ws=str(ws), verbosity_level=0)
gwf = sim.get_model()
nper = sim.tdis.nper.data
print(f"stress periods: {nper}")

### The wells and the observation point

Collect each well cell and the rate it pumps in every stress period with
`well_rates()`. MODFLOW 6 carries a stress period's well list forward until it
is replaced, so a period with no entry repeats the previous rate.

The two production wells, *reilly* and *vc*, are each screened across layers 4
and 5, so they occupy two cells apiece. A third well — the *prediction* well —
starts pumping only in stress period 12. Those different start times are what
make the superposition worth checking.

In [ ]:
rates = adjh.well_rates(gwf, nper)
well_names = {
    (3, 5, 14): "reilly (layer 4)",
    (4, 5, 14): "reilly (layer 5)",
    (3, 32, 5): "vc (layer 4)",
    (4, 32, 5): "vc (layer 5)",
    (4, 34, 15): "prediction",
}
for cell, series in rates.items():
    on = np.nonzero(series)[0]
    first = on[0] + 1 if on.size else None
    print(
        f"  {well_names.get(cell, str(cell)):18s} {series[on[0]]:11.1f} ft3/d"
        f"   from period {first}"
    )

obs_cell = (4, 33, 14)  # aq15, the observation well used below

Map the wells and the observation point so their positions are clear. The
observation well sits between the *vc* well and the prediction well, so it
should respond to both.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 8), constrained_layout=True)
mm = flopy.plot.PlotMapView(model=gwf, ax=ax, layer=0)
mm.plot_grid(lw=0.2, color="0.85")
mm.plot_bc("RIV", color="tab:cyan")
mm.plot_ibound()

xc, yc = gwf.modelgrid.xcellcenters, gwf.modelgrid.ycellcenters
seen = set()
for (k, i, j), label in well_names.items():
    if (i, j) in seen:
        continue
    seen.add((i, j))
    ax.plot(xc[i, j], yc[i, j], "ko", ms=8)
    ax.annotate(
        label.split(" (")[0],
        (xc[i, j], yc[i, j]),
        textcoords="offset points",
        xytext=(8, 4),
        fontsize=8,
    )
ax.plot(
    xc[obs_cell[1], obs_cell[2]],
    yc[obs_cell[1], obs_cell[2]],
    "r^",
    ms=12,
    label="observation well",
)
ax.legend(loc="upper right", fontsize=8)
ax.set_title("Wells and the observation well")

## Ask for the head at the observation well at every time

Write one performance measure per stress period, each one the head in the
observation cell at that period. Twenty measures means twenty backward sweeps,
and each sweep returns the sensitivity of that period's head to the pumping rate
in *every* period — the full response the superposition needs.

In [ ]:
measures = {f"t{kper:02d}": [(kper, 0, *obs_cell, "head")] for kper in range(1, nper)}
adj_file = adjh.write_adj_file(ws, "drawdown.adj", measures)

adj = mf6adj.Mf6Adj(
    adj_file.name,
    str(lib_name),
    logging_level="WARNING",
    working_directory=str(ws),
)
adj.solve_forward_model()
adj.solve_adjoint()
adj.finalize()
print(f"solved {len(measures)} performance measures")

## Superpose the well responses

For each measure — that is, for each time — sum the sensitivity to every well
cell multiplied by the rate that cell pumped in that period. Keep a per-well
running total as well, so the contribution of each well can be plotted
separately. The sum gives the head *change*, so negate it to report drawdown.

In [ ]:
reconstructed = np.zeros(nper)
per_well = {cell: np.zeros(nper) for cell in rates}

for kper_pm in range(1, nper):
    kernels = adjh.period_sensitivity(ws, f"t{kper_pm:02d}", "wel6_q")
    for kper, sens in kernels.items():
        for cell, series in rates.items():
            contribution = -sens[cell] * series[kper]
            reconstructed[kper_pm] += contribution
            per_well[cell][kper_pm] += contribution

Read the simulated drawdown for comparison: the head with no pumping minus the
head with pumping, at the observation cell.

In [ ]:
head_on = flopy.utils.HeadFile(ws / "sv.hds").get_alldata()
head_off = flopy.utils.HeadFile(ws_off / "sv.hds").get_alldata()
simulated = head_off[:, *obs_cell] - head_on[:, *obs_cell]

years = np.arange(nper)
for kper in (1, 5, 11, 15, nper - 1):
    print(
        f"  period {kper + 1:2d}   simulated {simulated[kper]:7.4f} ft"
        f"   superposed {reconstructed[kper]:7.4f} ft"
    )

### Compare the reconstruction with the model

Plot the two drawdown histories on the same axes, with the difference below.

In [ ]:
fig, (ax, axd) = plt.subplots(
    2,
    1,
    figsize=(8, 6),
    sharex=True,
    height_ratios=(3, 1),
    constrained_layout=True,
)
ax.plot(years, simulated, "o-", color="k", label="simulated (two model runs)")
ax.plot(
    years,
    reconstructed,
    "s--",
    color="tab:red",
    ms=5,
    label="superposed adjoint sensitivities",
)
ax.set_ylabel("drawdown (ft)")
ax.invert_yaxis()
ax.legend()
ax.set_title("Drawdown at the observation well")

axd.axhline(0.0, color="0.6", lw=0.8)
axd.plot(years, reconstructed - simulated, "o-", color="tab:purple", ms=4)
axd.set_xlabel("stress period")
axd.set_ylabel("difference (ft)")

**What to look for.** The two curves lie on top of each other. Drawdown climbs
quickly once the production wells start in period 2, flattens as the system
approaches a new balance, then jumps again in period 12 when the prediction well
starts. The difference in the lower panel stays near a thousandth of a foot
against a drawdown of more than four feet — a few hundredths of a percent. That
is the check that the adjoint sensitivities really are the unit responses of
this model, and that the model behaves linearly in the pumping rates.

### Which well is responsible for the drawdown

Stack the per-well contributions. Because superposition is a sum, each well's
share can be read off directly.

In [ ]:
labels, series = [], []
for cell in sorted(per_well, key=lambda c: well_names.get(c, str(c))):
    labels.append(well_names.get(cell, str(cell)))
    series.append(per_well[cell])

fig, ax = plt.subplots(figsize=(8, 5), constrained_layout=True)
ax.stackplot(years, *series, labels=labels, alpha=0.85)
ax.plot(years, simulated, "k--", lw=1.5, label="simulated total")
ax.set_xlabel("stress period")
ax.set_ylabel("drawdown (ft)")
ax.legend(loc="upper left", fontsize=8)
ax.set_title("Contribution of each well to the drawdown")

**What to look for.** The *vc* well, the largest and closest of the production
wells, accounts for most of the early drawdown, and its two screened layers
contribute in proportion to their rates. The *reilly* well is smaller and
farther away, so its band is thin. The prediction well contributes nothing until
period 12 and then adds most of the remaining drawdown — the single largest
share by the end. Splitting a drawdown this way is exactly what superposition is
for, and it costs no extra model runs.

## Map the drawdown over the whole valley

The same idea extends from one observation point to the entire grid, using
**reciprocity**: the head change at point A caused by pumping at B equals the
head change at B caused by the same pumping at A. So one backward solve with the
performance measure placed *at a well* returns that well's drawdown response
everywhere in the model at once.

Run one measure per well cell at the final stress period, then superpose the
resulting maps using each well's rate.

In [ ]:
map_measures = {f"w{n}": [(nper - 1, 0, *cell, "head")] for n, cell in enumerate(rates)}
map_file = adjh.write_adj_file(ws, "wells.adj", map_measures)

adj = mf6adj.Mf6Adj(
    map_file.name,
    str(lib_name),
    logging_level="WARNING",
    working_directory=str(ws),
)
adj.solve_forward_model()
adj.solve_adjoint()
adj.finalize()

superposed_map = np.zeros_like(head_on[0])
for n, (cell, series) in enumerate(rates.items()):
    active = [kper for kper in range(nper) if series[kper] != 0.0]
    response = adjh.total_sensitivity(ws, f"w{n}", "wel6_q", periods=active)
    superposed_map += -response * series[active[0]]

Compute the simulated drawdown map for the same time, then plot the two maps and
their difference side by side.

In [ ]:
simulated_map = head_off[-1] - head_on[-1]
difference = superposed_map - simulated_map
layer = 4  # the layer the production wells are screened in

fig, axes = plt.subplots(1, 3, figsize=(13, 7), constrained_layout=True)
vmax = np.nanmax(np.abs(simulated_map[layer]))
levels = np.linspace(0.0, vmax, 11)

for ax, data, title in (
    (axes[0], simulated_map, "simulated (two model runs)"),
    (axes[1], superposed_map, "superposed adjoint sensitivities"),
):
    mm = flopy.plot.PlotMapView(model=gwf, ax=ax, layer=layer)
    cb = mm.plot_array(data[layer], cmap="viridis", vmin=0.0, vmax=vmax)
    mm.contour_array(data[layer], levels=levels, colors="w", linewidths=0.5)
    mm.plot_ibound()
    ax.set_title(title, fontsize=10)
    ax.set_xticks([])
    ax.set_yticks([])
plt.colorbar(cb, ax=axes[1], shrink=0.4, label="drawdown (ft)")

dmax = np.nanmax(np.abs(difference[layer]))
mm = flopy.plot.PlotMapView(model=gwf, ax=axes[2], layer=layer)
cbd = mm.plot_array(difference[layer], cmap="RdBu_r", vmin=-dmax, vmax=dmax)
mm.plot_ibound()
axes[2].set_title("difference", fontsize=10)
axes[2].set_xticks([])
axes[2].set_yticks([])
plt.colorbar(cbd, ax=axes[2], shrink=0.4, label="difference (ft)")

print(f"largest drawdown:   {vmax:.3f} ft")
print(f"largest difference: {dmax:.3e} ft")

**What to look for.** The first two panels are the same picture: three cones of
depression, deepest at the *vc* and prediction wells, merging into one broad
depression across the southern half of the valley and flattening against the
river, which holds heads up. The third panel is the difference, and its color
scale is several orders of magnitude smaller than the drawdown itself — the
reconstruction is not merely similar, it is the same map.

The cost is worth noting. The simulated map needed two full model runs. The
superposed map needed one backward solve per well, and in exchange it can be
rebuilt for any set of pumping rates without running the model again. That is
what makes this approach useful for screening pumping scenarios: change the
rates, re-add the responses, and read the answer.

## Recap

- The adjoint sensitivity of a head to a well's rate is that well's **unit
  response** — the drawdown it causes per unit of pumping.
- On an aquifer that meets the **Theis** assumptions, the superposed
  sensitivities, the simulated drawdown, and the analytical solution all
  agree — the first two to machine precision, and the third to the model's
  own discretization error.
- Multiplying each well's response by its rate and summing over wells and stress
  periods rebuilds the drawdown history, matching the simulated drawdown to a few
  hundredths of a percent.
- Because the result is a sum, it separates cleanly into the share of drawdown
  each well is responsible for.
- **Reciprocity** — putting the measure at the well instead of the observation
  point — turns one backward solve per well into a drawdown map over the whole
  model.
- Once the responses are computed, any combination of pumping rates can be
  evaluated without running MODFLOW 6 again.